# NC-Musical: AI Music Transcription (Batch Colab GPU Pipeline)
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine directly inside Google Colab.

**Hardware Acceleration:** NVIDIA GPUs (T4, L4, A100)

**Key Features:**
- **Batch Transcription:** Process single or multiple YouTube URLs / uploaded files sequentially.
- **Original Song Naming:** Automatically names MIDI and chord files after the original YouTube song title or uploaded filename.
- **Dynamic Prompts:** No hardcoded links — easily paste links or be prompted on run.
- **Future-Proof yt-dlp:** Direct live master-branch builds for dependable extraction.
- **SoundFont Previews:** In-notebook audio playback via `MS Basic.sf3`.
- **Automated Chord Analysis:** Extracts and timestamps harmonic progressions using `music21`.
- **Automatic Downloads & ZIP:** Triggers immediate browser downloads and bundles batch results into `.zip`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [1]:
# @title 1. (Optional) Connect Google Drive for Persistent Model Storage
# @markdown Connecting Google Drive saves downloaded AI models and SoundFonts permanently so you don't re-download them across Colab restarts.
# @markdown Set **use_google_drive** to `False` to skip the authorization prompt and use fast session temporary storage instead.
use_google_drive = True # @param {type:"boolean"}
import os

if use_google_drive:
    if os.path.exists('/content/drive/MyDrive'):
        print("✓ Google Drive is already connected and active!")
    else:
        try:
            from google.colab import drive
            print("Connecting to Google Drive...")
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"Notice: Google Drive mount skipped ({e}). Falling back to local session storage.")

if os.path.exists('/content/drive/MyDrive'):
    drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
    os.makedirs(drive_models_dir, exist_ok=True)
    hf_cache_dir = os.path.join(drive_models_dir, "huggingface")
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    os.environ["TORCH_HOME"] = os.path.join(drive_models_dir, "torch")
    print(f"✓ Persistent Model Storage enabled: {drive_models_dir}")
else:
    local_cache_dir = "/content/cache/huggingface"
    os.makedirs(local_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = local_cache_dir
    print("ℹ Session Storage active: Models will be cached in /content/cache/ for this session.")


Connecting to Google Drive...
Mounted at /content/drive
✓ Persistent Model Storage enabled: /content/drive/MyDrive/NC-Musical-Models


In [2]:
# @title 2. Check Hardware (GPU / TPU v5e-1), Tokens & Setup Repository
import os
import sys
import subprocess
import getpass
import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Checking hardware accelerator environment...")
has_gpu = torch.cuda.is_available()
has_tpu = False

if has_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ NVIDIA GPU Detected: {gpu_name}")
    !nvidia-smi
else:
    # Check for Google Colab TPU v5e / TPU v2/v3 environment
    if 'COLAB_TPU_ADDR' in os.environ or os.environ.get('PJRT_DEVICE') == 'TPU' or os.path.exists('/dev/accel0'):
        has_tpu = True
        os.environ['PJRT_DEVICE'] = 'TPU'
        print('✓ Google Cloud TPU (v5e-1 / PJRT) Detected!')
    else:
        try:
            import torch_xla.core.xla_model as xm
            has_tpu = True
            print(f'✓ Google Cloud TPU Detected: {xm.xla_device()}')
        except Exception:
            print('ℹ Standard CPU environment detected (No GPU/TPU found).')

# Configure HuggingFace Token (HF_TOKEN)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    print("\nNotice: MuScriptor models on HuggingFace require a HuggingFace User Token (HF_TOKEN).")
    print("If saved in Colab Secrets as 'HF_TOKEN', it is loaded automatically. Otherwise, enter below:")
    try:
        hf_token = getpass.getpass("HuggingFace Token (HF_TOKEN): ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HuggingFace Access Token configured successfully!")

print("\nInstalling system dependencies (FFmpeg, FluidSynth, and Tor IP-Masking Daemon)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth tor
!service tor start || /etc/init.d/tor start || true

print("\nCloning / Updating NC-Musical repository...")
repo_dir = "/content/NC-Musical"
if not os.path.exists(repo_dir):
    res = subprocess.run(["git", "clone", "https://github.com/AgentHitmanFaris/NC-Musical.git", repo_dir], capture_output=True, text=True)
    if res.returncode != 0:
        print("Public clone failed (Repository is private). Access Token required.")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None

        if not token:
            print("\nPlease enter your GitHub Personal Access Token (PAT) to clone the private repository:")
            token = getpass.getpass("GitHub Token: ")

        token = token.strip()
        !git clone https://{token}@github.com/AgentHitmanFaris/NC-Musical.git /content/NC-Musical

if os.path.exists(repo_dir):
    %cd /content/NC-Musical
    !git checkout Stable 2>/dev/null || true
    !git pull origin Stable || !git pull || true
    print("\nInstalling Python dependencies (Future-proof live yt-dlp master branch)...")
    !pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" bgutil-ytdlp-pot-provider demucs music21 soundfile mido
    if has_tpu:
        try:
            import torch_xla
        except ImportError:
            print("Installing torch_xla TPU support...")
            !pip install -q torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html
    !pip install -q muscriptor || true

    print("\nRunning patch script...")
    !python patch_muscriptor.py || true
else:
    print("Error: Could not clone repository.")


Checking hardware accelerator environment...
✓ NVIDIA GPU Detected: Tesla T4
Wed Aug 26 05:23:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                    

In [3]:
# @title 3. Setup MS Basic.sf3 SoundFont
import os
import shutil
import urllib.request

local_sf3_path = "/content/NC-Musical/MS Basic.sf3"
drive_sf3_path = "/content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3"
has_drive = os.path.exists("/content/drive/MyDrive")

target_sf3 = drive_sf3_path if has_drive else local_sf3_path

if not os.path.exists(target_sf3):
    print("Downloading MS Basic.sf3 SoundFont (~50MB)...")
    sf3_urls = [
        "https://raw.githubusercontent.com/musescore/MuseScore/v4.1.0/share/sound/MS%20Basic.sf3",
        "https://huggingface.co/MuScriptor/assets/resolve/main/MuseScore_General.sf3"
    ]
    downloaded = False
    for url in sf3_urls:
        try:
            os.makedirs(os.path.dirname(target_sf3), exist_ok=True)
            urllib.request.urlretrieve(url, target_sf3)
            print(f"✓ MS Basic.sf3 downloaded successfully to {target_sf3}!")
            downloaded = True
            break
        except Exception as e:
            print(f"Notice ({e}): trying next source...")
    if not downloaded:
        print("Warning: SoundFont download failed.")
else:
    print(f"✓ MS Basic.sf3 SoundFont found at: {target_sf3}")

# Sync to local workspace path if Drive was used
if has_drive and os.path.exists(drive_sf3_path) and not os.path.exists(local_sf3_path):
    shutil.copy2(drive_sf3_path, local_sf3_path)


✓ MS Basic.sf3 SoundFont found at: /content/drive/MyDrive/NC-Musical-Models/MS Basic.sf3


In [ ]:
# @title 4. AI Music Transcription (Batch & Single Processing)
# @markdown ### 📥 Audio Input & Model Configuration
input_source = "Upload Audio/Video File(s)" # @param ["YouTube URL(s)", "Upload Audio/Video File(s)"]
youtube_urls = "" # @param {type:"string"}
model_size = "large" # @param ["small", "medium", "large"]

# @markdown ### 🎛️ Transcription Accuracy & Instrument Separation (Demucs AI Stems)
# @markdown High Accuracy isolates stems with Demucs (Vocal, Bass, Drums, Guitars/Keys) for Songsterr-clean separation with 0 bleed:
transcription_accuracy = "High Accuracy (Demucs AI Stem Separation - Clean 0 Bleed)" # @param ["High Accuracy (Demucs AI Stem Separation - Clean 0 Bleed)", "Standard Fast (Direct Single-Pass Mix)"]
instrument_focus = "All Instruments (Full Multitrack)" # @param ["All Instruments (Full Multitrack)", "Vocals Only (Clean Solo Voice)", "Piano Only (Full Grand Staff)", "Guitars Only (Clean & Distortion)", "Rhythm Section (Bass & Drums)"]

# @markdown ### ⏱️ Musical Settings (Time Signature, BPM, Key & Triplets)
time_signature = "Auto-Detect" # @param ["Auto-Detect", "4/4", "3/4", "6/8", "12/8", "2/4", "5/4", "7/8"]
bpm = "Auto-Detect" # @param ["Auto-Detect", "80", "90", "100", "110", "120", "125", "130", "135", "140", "145", "150", "160", "Custom"]
custom_bpm = 120.0 # @param {type:"number"}
key_signature = "Auto-Detect" # @param ["Auto-Detect", "C", "G", "D", "A", "E", "B", "F#", "F", "Bb", "Eb", "Ab", "Db", "Gb", "Am", "Em", "Bm", "F#m", "C#m", "G#m", "Dm", "Gm", "Cm", "Fm", "Bbm", "Ebm"]
use_triplets = False # @param {type:"boolean"}

# @markdown ### 📤 Export Options
export_musicxml = True # @param {type:"boolean"}
export_chord_names = False # @param {type:"boolean"}
download_zip_for_batch = False # @param {type:"boolean"}

import os
import sys
import re
import io
import time
import math
import gc
import zipfile
import shutil
import subprocess
import torch
from pathlib import Path
from IPython.display import Audio, display
import google.colab.files

# Set PyTorch memory allocator to avoid fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Ensure yt-dlp, music21, and mido are present
!pip install -U -q --no-cache-dir "yt-dlp @ https://github.com/yt-dlp/yt-dlp/archive/master.tar.gz" music21 mido
import yt_dlp
import mido

def sanitize_filename(name: str) -> str:
    """Clean filename of illegal filesystem characters."""
    clean = re.sub(r'[\\/*?:"<>|]', "", name).strip()
    clean = re.sub(r'\s+', " ", clean)
    return clean[:120] if clean else "transcription_result"

# ── 1. Resolve Target Instruments & Accuracy Mode ────────────────────────────
inst_map = {
    "All Instruments (Full Multitrack)": None,
    "Vocals Only (Clean Solo Voice)": ["voice"],
    "Piano Only (Full Grand Staff)": ["acoustic_piano", "electric_piano"],
    "Guitars Only (Clean & Distortion)": ["clean_electric_guitar", "distorted_electric_guitar", "acoustic_guitar"],
    "Rhythm Section (Bass & Drums)": ["electric_bass", "drums"]
}
inst_list = inst_map.get(instrument_focus, None)
is_high_accuracy = transcription_accuracy.startswith("High Accuracy")

print(f"🎯 Accuracy Mode: {'High Accuracy (Demucs Decoupled Stems)' if is_high_accuracy else 'Standard Fast (Direct Mix)'}")
print(f"🎛️ Instrument Focus: {instrument_focus}")

def separate_stems_demucs(audio_file, out_dir="/content/stems", two_stems=False):
    """Isolate audio stems using Hybrid Transformer Demucs v4."""
    out_p = Path(out_dir)
    out_p.mkdir(parents=True, exist_ok=True)
    cmd = ["demucs", "-n", "htdemucs", "-d", "cuda" if torch.cuda.is_available() else "cpu", "--out", str(out_p)]
    if two_stems:
        cmd.extend(["--two-stems", "vocals"])
    cmd.append(str(audio_file))
    print(f"🎸 Isolating audio stems with Demucs ({'Vocals + Backing' if two_stems else 'Vocals, Bass, Drums, Other'})...")
    subprocess.run(cmd, check=True)
    stem_folder = out_p / "htdemucs" / Path(audio_file).stem
    return stem_folder

# ── 2. Timing, Meter, Key & Triplets Setup ─────────────────────────────────────
def parse_time_signature_choice(ts_str):
    if not ts_str or "auto" in str(ts_str).lower():
        return None, None
    clean = str(ts_str).split("(")[0].strip()
    if "/" in clean:
        parts = clean.split("/")
        try:
            return int(parts[0].strip()), int(parts[1].strip())
        except ValueError:
            pass
    return 4, 4

def detect_time_sig_from_midi_notes(mid):
    """Auto-detect time signature (numerator, denominator) from MIDI tracks or onset periodicity."""
    for track in mid.tracks:
        for msg in track:
            if msg.type == 'time_signature' and getattr(msg, 'numerator', 0) > 0:
                return int(msg.numerator), int(msg.denominator)
    ticks_per_beat = mid.ticks_per_beat or 480
    onsets_beats = []
    for track in mid.tracks:
        curr = 0
        for msg in track:
            curr += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                onsets_beats.append(curr / ticks_per_beat)
    if len(onsets_beats) < 12:
        return 4, 4
    score_3 = sum(1 for b in onsets_beats if abs(b % 3.0) < 0.15 or abs(3.0 - (b % 3.0)) < 0.15)
    score_4 = sum(1 for b in onsets_beats if abs(b % 4.0) < 0.15 or abs(4.0 - (b % 4.0)) < 0.15)
    if score_3 > score_4 * 1.35:
        return 3, 4
    return 4, 4

grid_div = 3 if use_triplets else 4
is_no_tuplets = not use_triplets
enable_quantization = True
quantize_strength = 1.0
fill_gaps_beats = 0.25
trim_leading_bars = False
separate_instrument_tracks = True

print(f"⏱️ Time Signature: {time_signature} | BPM: {bpm} | Key: {key_signature} | Triplets: {'Enabled (Swing/Tuplets)' if use_triplets else 'Disabled (Straight 1/16 Grid)'}")

# ── 3. Embedded High-Performance MIDI Engine ──────────────────────────────────
_MAJOR_PROFILE = [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]
_MINOR_PROFILE = [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]
_MIDO_MAJOR_KEYS = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'F#', 'G', 'Ab', 'A', 'Bb', 'B']
_MIDO_MINOR_KEYS = ['Cm', 'C#m', 'Dm', 'Ebm', 'Em', 'Fm', 'F#m', 'Gm', 'G#m', 'Am', 'Bbm', 'Bm']
_KEY_PRESET_MAP = {
    "c major / a minor": "C", "g major / e minor (1#)": "G", "d major / b minor (2#)": "D",
    "a major / f# minor (3#)": "A", "e major / c# minor (4#)": "E", "b major / g# minor (5#)": "B",
    "f# major / d# minor (6#)": "F#", "f major / d minor (1b)": "F", "bb major / g minor (2b)": "Bb",
    "eb major / c minor (3b)": "Eb", "ab major / f minor (4b)": "Ab", "db major / bb minor (5b)": "Db",
    "gb major / eb minor (6b)": "Gb"
}

def detect_key_signature_from_notes(mid):
    """Accurately detect key signature using pitch histogram correlation."""
    try:
        from music21 import converter
        temp_buf = io.BytesIO()
        mid.save(file=temp_buf)
        temp_buf.seek(0)
        s = converter.parse(temp_buf.getvalue(), format='midi')
        k = s.analyze('key')
        tonic = k.tonic.name
        return tonic + ('m' if k.mode == 'minor' else '')
    except Exception:
        pass

    histogram = [0.0] * 12
    ticks_per_beat = mid.ticks_per_beat or 480
    for track in mid.tracks:
        active = {}
        curr = 0
        for msg in track:
            curr += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                active[msg.note] = curr
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                if msg.note in active:
                    start = active.pop(msg.note)
                    histogram[msg.note % 12] += max(1, curr - start) / ticks_per_beat

    if sum(histogram) == 0:
        return 'C'

    def corr(x, y):
        n = 12
        mx, my = sum(x) / n, sum(y) / n
        num = sum((x[i] - mx) * (y[i] - my) for i in range(n))
        den = math.sqrt(sum((x[i] - mx) ** 2 for i in range(n)) * sum((y[i] - my) ** 2 for i in range(n)))
        return num / den if den > 0 else 0

    best_score, best_key = -float('inf'), 'C'
    for r in range(12):
        maj_r = _MAJOR_PROFILE[-r:] + _MAJOR_PROFILE[:-r]
        min_r = _MINOR_PROFILE[-r:] + _MINOR_PROFILE[:-r]
        c_maj = corr(histogram, maj_r)
        if c_maj > best_score:
            best_score, best_key = c_maj, _MIDO_MAJOR_KEYS[r]
        c_min = corr(histogram, min_r)
        if c_min > best_score:
            best_score, best_key = c_min, _MIDO_MINOR_KEYS[r]
    return best_key

def separate_multichannel_tracks(mid: mido.MidiFile) -> mido.MidiFile:
    """Separate MIDI channels and instruments into dedicated staves for MuseScore.
    
    1. Piano is partitioned into a complete Grand Staff:
       - 'Piano (Right Hand - Treble Clef)': Notes >= 60 (Treble staff with full polyphonic chords)
       - 'Piano (Left Hand - Bass Clef)': Notes < 60 (Bass staff with full polyphonic chords)
    2. Drums are assigned to Channel 9 with full polyphonic kit (Kick, Snare, Hi-Hat, etc.).
    3. Guitars are separated into distinct tracks:
       - 'Distorted Electric Guitar' (Overdrive/Distortion)
       - 'Clean Electric Guitar' (or Acoustic Guitar)
       - 'Electric Bass' (Bass Clef)
    4. Vocals are assigned to a dedicated Track: 'Vocals (Clean Voice 1)'.
    5. Strings, Brass, Woodwinds, and Synths are assigned to their own dedicated staves.
    """
    new_mid = mido.MidiFile(ticks_per_beat=mid.ticks_per_beat or 480)
    
    # Track 0 preserves global meta messages (tempo, meter, key)
    tempo_track = mido.MidiTrack()
    new_mid.tracks.append(tempo_track)
    
    # Track active program per channel and gather all note pairs & non-note events
    active_program = {}
    note_events_by_group = {}
    other_events = []
    
    # First pass: map tempo track meta messages
    for track in mid.tracks:
        curr_tick = 0
        for msg in track:
            curr_tick += msg.time
            if msg.is_meta and msg.type in ('set_tempo', 'time_signature', 'key_signature', 'smpte_offset'):
                tempo_track.append(msg.copy(time=curr_tick))
                
    tempo_track_events = sorted([(msg.time, msg) for msg in tempo_track], key=lambda x: x[0])
    tempo_track.clear()
    prev = 0
    for tick, msg in tempo_track_events:
        msg.time = max(0, tick - prev)
        tempo_track.append(msg)
        prev = tick
    tempo_track.append(mido.MetaMessage('end_of_track', time=0))

    # Second pass: extract note pairs with instrument classification
    for track in mid.tracks:
        curr_tick = 0
        active_notes = {}
        track_meta_name = ""
        for msg in track:
            if msg.is_meta and msg.type == 'track_name':
                track_meta_name = msg.name.lower()
                break

        for msg in track:
            curr_tick += msg.time
            if msg.is_meta:
                continue
            ch = getattr(msg, 'channel', 0)
            if msg.type == 'program_change':
                active_program[ch] = msg.program
            elif msg.type == 'note_on' and msg.velocity > 0:
                prog = active_program.get(ch, None)
                active_notes.setdefault((ch, msg.note), []).append((curr_tick, msg, prog, track_meta_name))
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                k = (ch, msg.note)
                if k in active_notes and active_notes[k]:
                    start_t, on_msg, prog, tname = active_notes[k].pop(0)
                    
                    # Instrument classification
                    if ch == 9 or "drum" in tname:
                        grp = 'drums'
                    elif "vocal" in tname or "voice" in tname or prog in (52, 53, 54, 85, 91) or (ch == 0 and (prog is None or prog in (52, 53, 54, 85, 91)) and "piano" not in tname and "guitar" not in tname):
                        grp = 'vocal'
                    elif "piano" in tname or "keys" in tname or (prog in (0, 1, 2, 3, 4, 5) and ch != 0) or (prog in (0, 1, 2, 3, 4, 5) and "piano" in tname):
                        grp = 'piano'
                    elif "bass" in tname or prog in (32, 33, 34, 35, 36, 37, 38, 39):
                        grp = 'bass'
                    elif "distort" in tname or prog in (29, 30):
                        grp = 'dist_guitar'
                    elif "guitar" in tname or prog in (24, 25, 26, 27, 28):
                        grp = 'clean_guitar'
                    elif prog is not None and 40 <= prog <= 51:
                        grp = 'strings'
                    elif prog is not None and 56 <= prog <= 63:
                        grp = 'brass'
                    elif prog is not None and 64 <= prog <= 75:
                        grp = 'woodwinds'
                    elif prog is not None and 80 <= prog <= 103:
                        grp = 'synth'
                    elif prog in (0, 1, 2, 3, 4, 5):
                        grp = 'piano'
                    else:
                        grp = 'vocal' if ch == 0 else f'inst_{prog or ch}'

                    note_events_by_group.setdefault(grp, []).append({
                        'channel': ch,
                        'program': prog or 0,
                        'pitch': msg.note,
                        'start': start_t,
                        'end': curr_tick,
                        'on_msg': on_msg,
                        'off_msg': msg
                    })

    # If no structured groups found, fallback to basic tracks
    if not note_events_by_group:
        empty_tr = mido.MidiTrack()
        empty_tr.append(mido.MetaMessage('track_name', name="Lead Melody", time=0))
        empty_tr.append(mido.MetaMessage('end_of_track', time=0))
        new_mid.tracks.append(empty_tr)
        return new_mid

    # Helper to serialize note pairs into a new MidiTrack
    def _create_track(title, channel, program, notes):
        if not notes:
            return None
        tr = mido.MidiTrack()
        tr.append(mido.MetaMessage('track_name', name=title, time=0))
        tr.append(mido.Message('program_change', channel=channel, program=program, time=0))
        events = []
        for n in notes:
            on_m = n['on_msg'].copy(channel=channel)
            off_m = n['off_msg'].copy(channel=channel)
            events.append((n['start'], 2, on_m))
            events.append((n['end'], 1, off_m))
        events.sort(key=lambda x: (x[0], x[1]))
        prev_t = 0
        for atick, _, m in events:
            m.time = max(0, atick - prev_t)
            tr.append(m)
            prev_t = atick
        tr.append(mido.MetaMessage('end_of_track', time=0))
        return tr

    # 1. Vocals Track (Channel 0, Program 54)
    if 'vocal' in note_events_by_group:
        tr = _create_track("Vocals (Clean Voice 1)", 0, 54, note_events_by_group['vocal'])
        if tr: new_mid.tracks.append(tr)

    # 2. Piano Grand Staff: Treble Clef (RH >= 60) and Bass Clef (LH < 60)
    if 'piano' in note_events_by_group:
        piano_notes = note_events_by_group['piano']
        rh_notes = [n for n in piano_notes if n['pitch'] >= 60]
        lh_notes = [n for n in piano_notes if n['pitch'] < 60]

        if rh_notes:
            tr_rh = _create_track("Piano (Right Hand - Treble Clef)", 1, 0, rh_notes)
            if tr_rh: new_mid.tracks.append(tr_rh)
        if lh_notes:
            tr_lh = _create_track("Piano (Left Hand - Bass Clef)", 2, 0, lh_notes)
            if tr_lh: new_mid.tracks.append(tr_lh)
        if not rh_notes and not lh_notes:
            tr_p = _create_track("Piano (Full Grand Staff)", 1, 0, piano_notes)
            if tr_p: new_mid.tracks.append(tr_p)

    # 3. Clean / Acoustic Guitar (Channel 3, Program 27/25)
    if 'clean_guitar' in note_events_by_group:
        tr = _create_track("Clean Electric Guitar", 3, 27, note_events_by_group['clean_guitar'])
        if tr: new_mid.tracks.append(tr)

    # 4. Distorted Electric Guitar (Channel 4, Program 29)
    if 'dist_guitar' in note_events_by_group:
        tr = _create_track("Distorted Electric Guitar", 4, 29, note_events_by_group['dist_guitar'])
        if tr: new_mid.tracks.append(tr)

    # 5. Electric Bass (Channel 5, Program 33 - Bass Clef)
    if 'bass' in note_events_by_group:
        tr = _create_track("Electric Bass", 5, 33, note_events_by_group['bass'])
        if tr: new_mid.tracks.append(tr)

    # 6. Strings (Channel 6, Program 48)
    if 'strings' in note_events_by_group:
        tr = _create_track("Strings Ensemble", 6, 48, note_events_by_group['strings'])
        if tr: new_mid.tracks.append(tr)

    # 7. Brass (Channel 7, Program 61)
    if 'brass' in note_events_by_group:
        tr = _create_track("Brass Section", 7, 61, note_events_by_group['brass'])
        if tr: new_mid.tracks.append(tr)

    # 8. Woodwinds / Flutes (Channel 8, Program 73)
    if 'woodwinds' in note_events_by_group:
        tr = _create_track("Woodwinds / Flutes", 8, 73, note_events_by_group['woodwinds'])
        if tr: new_mid.tracks.append(tr)

    # 9. Synthesizer / Pads (Channel 10, Program 80)
    if 'synth' in note_events_by_group:
        tr = _create_track("Synthesizer", 10, 80, note_events_by_group['synth'])
        if tr: new_mid.tracks.append(tr)

    # 10. Drums (Channel 9, Program 0 - GM Standard Kit)
    if 'drums' in note_events_by_group:
        tr = _create_track("Drums (Full Kit)", 9, 0, note_events_by_group['drums'])
        if tr: new_mid.tracks.append(tr)

    # Any remaining unmapped groups
    for grp, notes in note_events_by_group.items():
        if grp not in ('vocal', 'piano', 'clean_guitar', 'dist_guitar', 'bass', 'strings', 'brass', 'woodwinds', 'synth', 'drums'):
            prog = notes[0].get('program', 0)
            ch = min(15, max(11, notes[0].get('channel', 11)))
            tr = _create_track(f"Instrument (Track {grp})", ch, prog, notes)
            if tr: new_mid.tracks.append(tr)

    return new_mid


def clean_monophonic_vocal(
    note_pairs: list[dict],
    grid_ticks: int = 120,
    fill_gap_ticks: int = 120
) -> list[dict]:
    """Fine-tuned vocal melody cleaner: produces the cleanest single-voice notation and structure.
    
    1. Rejects spurious transient blips (< 24 ticks / ~30ms) caused by unvoiced consonants/breath noise.
    2. Filters non-vocal acoustic rumble (< 36) and high overtone squeaks (> 90).
    3. Resolves polyphonic overtone collisions: selects the dominant fundamental pitch
       (favoring higher acoustic velocity and melodic continuity within singing range).
    4. Enforces strict zero-overlap: note[i].end <= note[i+1].start (no 2-voicing in MuseScore).
    5. Applies natural legato smoothing: connects syllables across small breath gaps (<= 1/16th)
       while preserving natural pauses.
    """
    if not note_pairs:
        return []

    min_dur = max(24, grid_ticks // 4)
    # Sort chronologically by start time; for same onset, higher velocity and singing-range pitch first
    def sort_key(n):
        vel = getattr(n.get('on_msg'), 'velocity', 64)
        p = n['pitch']
        in_vocal_range = 1 if 40 <= p <= 85 else 0
        return (n['start'], -vel, -in_vocal_range, -p)

    sorted_notes = sorted(note_pairs, key=sort_key)
    chain = []
    
    for n in sorted_notes:
        dur = n['end'] - n['start']
        # Filter micro-jitter blips
        if dur < min_dur:
            continue
        # Filter out-of-range vocal noise if outside standard vocal compass
        if n['pitch'] < 36 or n['pitch'] > 92:
            continue

        if not chain:
            chain.append(dict(n))
            continue

        prev = chain[-1]
        
        # Same onset collision: pick the dominant fundamental vocal note
        if n['start'] == prev['start']:
            n_vel = getattr(n.get('on_msg'), 'velocity', 64)
            prev_vel = getattr(prev.get('on_msg'), 'velocity', 64)
            if n_vel > prev_vel + 10:
                chain[-1] = dict(n)
            elif abs(n_vel - prev_vel) <= 10:
                # Prefer central vocal register (A2 to A5, 45 to 81)
                if 45 <= n['pitch'] <= 81 and (prev['pitch'] < 45 or prev['pitch'] > 81):
                    chain[-1] = dict(n)
                elif n['pitch'] > prev['pitch'] and 45 <= n['pitch'] <= 84:
                    chain[-1] = dict(n)
            continue

        # If previous note extends past the start of the new note, truncate previous note!
        if prev['end'] > n['start']:
            prev['end'] = n['start']
            if prev['end'] - prev['start'] < min_dur:
                chain.pop()
                if chain and chain[-1]['end'] > n['start']:
                    chain[-1]['end'] = n['start']

        chain.append(dict(n))

    # Natural legato smoothing between consecutive syllables
    if fill_gap_ticks > 0:
        for i in range(len(chain) - 1):
            cur_n = chain[i]
            next_n = chain[i + 1]
            gap = next_n['start'] - cur_n['end']
            if 0 < gap <= fill_gap_ticks:
                cur_n['end'] = next_n['start']

    cleaned = [n for n in chain if n['end'] > n['start']]
    for i in range(len(cleaned) - 1):
        if cleaned[i]['end'] > cleaned[i + 1]['start']:
            cleaned[i]['end'] = cleaned[i + 1]['start']

    return cleaned


def clean_monophonic_voice(
    note_pairs: list[dict],
    grid_ticks: int = 120,
    fill_gap_ticks: int = 120
) -> list[dict]:
    """General monophonic voice reduction (for Bass and Lead lines)."""
    return clean_monophonic_vocal(note_pairs, grid_ticks=grid_ticks, fill_gap_ticks=fill_gap_ticks)


def quantize_midi_tracks(
    mid: mido.MidiFile,
    grid_div: int = 4,
    quantize_strength: float = 1.0,
    fill_gaps_beats: float = 0.0,
    trim_leading_bars: bool = False,
    beats_per_bar: int = 4,
    time_sig_denom: int = 4,
    single_voice: bool = True,
    separate_channels: bool = True
) -> mido.MidiFile:
    """Snap note onsets and durations to the specified beat grid and eliminate multi-voicing.
    
    grid_div=4: sixteenth notes (no tuplets)
    grid_div=2: eighth notes (no tuplets)
    grid_div=8: thirty-second notes (no tuplets)
    grid_div=3: eighth triplets
    grid_div=6: sixteenth triplets
    
    single_voice=True: Enforces strictly 1 clean voice (Voice 1 in MuseScore) with ZERO overlaps.
    separate_channels=True: Splits channels into dedicated tracks so MuseScore creates clean individual staves.
    """
    ticks_per_beat = mid.ticks_per_beat or 480
    ticks_per_quarter = ticks_per_beat
    ticks_per_beat_unit = int(round(ticks_per_quarter * (4.0 / time_sig_denom))) if time_sig_denom else ticks_per_beat

    if not grid_div or grid_div <= 0:
        grid_div = 4

    grid_ticks = max(1, int(round(ticks_per_beat_unit / grid_div)))
    fill_gap_ticks = int(round(fill_gaps_beats * ticks_per_beat_unit)) if fill_gaps_beats > 0 else 0
    ticks_per_bar = max(grid_ticks, int(round(ticks_per_beat_unit * beats_per_bar)))

    # Step 1: Optionally separate channels to avoid multi-instrument collisions in a single staff
    working_mid = separate_multichannel_tracks(mid) if separate_channels else mid

    # Step 2: Check leading empty bars
    shift_ticks = 0
    if trim_leading_bars:
        earliest_note_tick = float('inf')
        for track in working_mid.tracks:
            curr = 0
            for msg in track:
                curr += msg.time
                if msg.type == 'note_on' and msg.velocity > 0:
                    if curr < earliest_note_tick:
                        earliest_note_tick = curr
        if earliest_note_tick != float('inf') and earliest_note_tick >= ticks_per_bar:
            shift_bars = int(earliest_note_tick // ticks_per_bar)
            shift_ticks = shift_bars * ticks_per_bar

    # Step 3: Quantize and clean each track
    for t_idx, track in enumerate(working_mid.tracks):
        curr_tick = 0
        abs_events = []
        for msg in track:
            curr_tick += msg.time
            abs_events.append((curr_tick, msg))

        active_notes = {}
        note_pairs = []
        other_events = []

        for atick, msg in abs_events:
            if msg.type == 'note_on' and msg.velocity > 0:
                key = (msg.channel, msg.note)
                active_notes.setdefault(key, []).append((atick, msg))
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                key = (msg.channel, msg.note)
                if key in active_notes and active_notes[key]:
                    start_tick, on_msg = active_notes[key].pop(0)
                    note_pairs.append({
                        'channel': msg.channel,
                        'pitch': msg.note,
                        'start': start_tick,
                        'end': atick,
                        'on_msg': on_msg,
                        'off_msg': msg
                    })
                else:
                    other_events.append((atick, msg))
            else:
                other_events.append((atick, msg))

        # Close any lingering open notes
        for key, unclosed in active_notes.items():
            for start_tick, on_msg in unclosed:
                note_pairs.append({
                    'channel': key[0],
                    'pitch': key[1],
                    'start': start_tick,
                    'end': start_tick + grid_ticks,
                    'on_msg': on_msg,
                    'off_msg': mido.Message('note_off', channel=key[0], note=key[1], velocity=0, time=0)
                })

        if not note_pairs:
            continue

        # Extract track metadata to determine instrument type
        track_name = ""
        for msg in track:
            if msg.is_meta and msg.type == 'track_name':
                track_name = msg.name
                break
        tname_lower = track_name.lower()

        is_piano_track = ("piano" in tname_lower) or ("grand staff" in tname_lower)
        is_drums_track = ("drum" in tname_lower) or any(getattr(m, 'channel', 0) == 9 for m in track if not m.is_meta)
        is_guitar_track = ("guitar" in tname_lower) and ("bass" not in tname_lower)
        is_bass_track = ("bass" in tname_lower) and not is_drums_track and not is_piano_track
        is_other_polyphonic = ("strings" in tname_lower) or ("brass" in tname_lower) or ("woodwinds" in tname_lower) or ("synth" in tname_lower)

        is_polyphonic = is_piano_track or is_drums_track or is_guitar_track or is_other_polyphonic
        is_vocal_track = ("vocal" in tname_lower) or ("voice" in tname_lower) or (not is_polyphonic and not is_bass_track)

        # Voicing strategy:
        # 1. Vocals: Strict fine-tuned monophonic reduction (Voice 1 in MuseScore, zero 2-voicing)
        # 2. Bass: Monophonic bassline reduction (clean bass clef)
        # 3. Piano, Drums, Guitars: PRESERVE FULL POLYPHONY (Chords, Full Drum Kit, Both Piano Hands)
        if is_vocal_track and single_voice:
            note_pairs = clean_monophonic_vocal(
                note_pairs,
                grid_ticks=grid_ticks,
                fill_gap_ticks=fill_gap_ticks
            )
        elif is_bass_track and single_voice:
            note_pairs = clean_monophonic_voice(
                note_pairs,
                grid_ticks=grid_ticks,
                fill_gap_ticks=fill_gap_ticks
            )
        else:
            note_pairs.sort(key=lambda n: (n['channel'], n['pitch'], n['start']))
            # Fill small rest gaps between consecutive notes of the same pitch
            if fill_gap_ticks > 0:
                for i in range(len(note_pairs) - 1):
                    cur_n = note_pairs[i]
                    next_n = note_pairs[i + 1]
                    if cur_n['channel'] == next_n['channel'] and cur_n['pitch'] == next_n['pitch']:
                        gap = next_n['start'] - cur_n['end']
                        if 0 < gap <= fill_gap_ticks:
                            cur_n['end'] = next_n['start']

        # Resolve BPM (auto or manual)
        final_bpm = 120.0
        if "auto" in str(bpm).lower():
            try:
                print("Attempting AI beat tracking & tempo detection...")
                if hasattr(model, 'transcribe_and_postprocess'):
                    _, ai_grid = model.transcribe_and_postprocess(wav_input, detect_tempo="best-effort", quantize=False)
                    if ai_grid is not None and getattr(ai_grid, 'bpm', None):
                        final_bpm = float(ai_grid.bpm)
                        print(f"✓ AI Detected Tempo: {final_bpm:.1f} BPM")
            except Exception as tempo_err:
                print(f"⚠️ AI beat-tracking notice: {tempo_err} -> Using adaptive tempo {final_bpm:.1f} BPM")
        elif str(bpm).lower() == "custom":
            final_bpm = float(custom_bpm)
        else:
            try:
                final_bpm = float(bpm)
            except ValueError:
                final_bpm = 120.0

        raw_midi_bytes = model.events_to_midi_bytes(iter(events))
        raw_mid = mido.MidiFile(file=io.BytesIO(raw_midi_bytes))

        # Resolve Time Signature (auto or manual)
        req_num, req_denom = parse_time_signature_choice(time_signature)
        if req_num is not None:
            eff_ts_num, eff_ts_denom = req_num, req_denom
        else:
            eff_ts_num, eff_ts_denom = detect_time_sig_from_midi_notes(raw_mid)

        # Apply robust musical post-processing (Time Signature, Key Signature, Tempo, Single Voice & Quantization)
        print("Applying post-processing (Time Signature, Key Signature, Tempo, Single Voice & Quantization)...")
        processed_mid, final_midi_bytes, resolved_key = postprocess_midi_events(
            raw_mid,
            bpm_val=final_bpm,
            time_sig_num=eff_ts_num,
            time_sig_denom=eff_ts_denom,
            key_sig_choice=key_signature,
            quantize=enable_quantization,
            grid_division=grid_div,
            quant_strength=quantize_strength,
            fill_gaps=fill_gaps_beats,
            trim_lead_bars=trim_leading_bars,
            single_voice=True,
            separate_channels=separate_instrument_tracks
        )

import math
import xml.etree.ElementTree as ET
from pathlib import Path
import mido

_STEP_NAMES = ['C', 'C', 'D', 'D', 'E', 'F', 'F', 'G', 'G', 'A', 'A', 'B']
_ALTER_VALS = [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0]

_KEY_FIFTHS = {
    'C': 0, 'Am': 0,
    'G': 1, 'Em': 1,
    'D': 2, 'Bm': 2,
    'A': 3, 'F#m': 3,
    'E': 4, 'C#m': 4,
    'B': 5, 'G#m': 5,
    'F#': 6, 'D#m': 6,
    'F': -1, 'Dm': -1,
    'Bb': -2, 'Gm': -2,
    'Eb': -3, 'Cm': -3,
    'Ab': -4, 'Fm': -4,
    'Db': -5, 'Bbm': -5,
    'Gb': -6, 'Ebm': -6
}


def export_midi_to_musicxml(
    mid: mido.MidiFile,
    output_xml_path: str | Path,
    title: str = "Transcription",
    bpm: float = 120.0,
    time_sig_num: int = 4,
    time_sig_denom: int = 4,
    key_signature: str | None = "C"
) -> Path:
    """Export a multi-track MidiFile into a clean, well-formatted MusicXML file for MuseScore.
    
    Guarantees:
      - Grand Staff (Treble on top, Bass on bottom) for Piano.
      - Treble Clef & Voice 1 for Vocals (strictly no 2-voicing).
      - Percussion Clef on 5-line staff for Drums.
      - Bass Clef (F4) for Electric Bass.
      - Treble Clef for Guitars.
      - Exact bar-line measures with divisions, key, tempo, and meter.
    """
    ticks_per_beat = mid.ticks_per_beat or 480
    ticks_per_bar = max(ticks_per_beat, int(round(ticks_per_beat * (4.0 / time_sig_denom) * time_sig_num)))

    score = ET.Element('score-partwise', version="3.1")

    # Work & Title metadata
    work = ET.SubElement(score, 'work')
    ET.SubElement(work, 'work-title').text = title

    part_list = ET.SubElement(score, 'part-list')

    # Scan tracks to find active tracks with notes
    track_entries = []
    p_counter = 1

    for t_idx, track in enumerate(mid.tracks):
        # Extract track name
        tname = f"Track {t_idx}"
        for msg in track:
            if msg.is_meta and msg.type == 'track_name':
                tname = msg.name
                break

        # Check if track has note events
        has_notes = any((getattr(m, 'type', '') == 'note_on' and getattr(m, 'velocity', 0) > 0) for m in track if not m.is_meta)
        if not has_notes:
            continue

        pid = f"P{p_counter}"
        p_counter += 1

        score_part = ET.SubElement(part_list, 'score-part', id=pid)
        ET.SubElement(score_part, 'part-name').text = tname

        track_entries.append((pid, tname, track))

    if not track_entries:
        # Fallback empty score
        pid = "P1"
        score_part = ET.SubElement(part_list, 'score-part', id=pid)
        ET.SubElement(score_part, 'part-name').text = "Score"
        track_entries.append((pid, "Score", mido.MidiTrack()))

    fifths = _KEY_FIFTHS.get(key_signature, 0)
    mode = 'minor' if (key_signature and 'm' in key_signature and key_signature not in ('C', 'F', 'D', 'E', 'A', 'B')) else 'major'

    # Build XML part elements
    for pid, tname, track in track_entries:
        part = ET.SubElement(score, 'part', id=pid)

        tname_lower = tname.lower()
        is_bass = ("bass" in tname_lower) and not ("drum" in tname_lower) and not ("left hand" in tname_lower)
        is_piano_lh = ("left hand" in tname_lower) or ("bass clef" in tname_lower and "piano" in tname_lower)
        is_drums = ("drum" in tname_lower)
        is_vocal = ("vocal" in tname_lower) or ("voice" in tname_lower)

        # Extract notes with absolute tick times
        abs_events = []
        curr_tick = 0
        for msg in track:
            curr_tick += msg.time
            if not msg.is_meta:
                abs_events.append((curr_tick, msg))

        active = {}
        note_pairs = []
        for atick, msg in abs_events:
            if msg.type == 'note_on' and msg.velocity > 0:
                active.setdefault(msg.note, []).append((atick, msg))
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                if msg.note in active and active[msg.note]:
                    st, on_m = active[msg.note].pop(0)
                    note_pairs.append({
                        'pitch': msg.note,
                        'start': st,
                        'end': max(st + 1, atick),
                        'dur': max(1, atick - st),
                        'vel': on_m.velocity
                    })

        note_pairs.sort(key=lambda n: (n['start'], n['pitch']))

        # Determine total measures required
        max_tick = max([n['end'] for n in note_pairs], default=ticks_per_bar)
        num_measures = max(1, int(math.ceil(max_tick / ticks_per_bar)))

        # Group notes by measure
        measure_notes = {m_idx: [] for m_idx in range(1, num_measures + 1)}
        for n in note_pairs:
            m_idx = (n['start'] // ticks_per_bar) + 1
            if m_idx <= num_measures:
                rel_start = n['start'] % ticks_per_bar
                measure_notes[m_idx].append({
                    'pitch': n['pitch'],
                    'rel_start': rel_start,
                    'dur': min(n['dur'], ticks_per_bar - rel_start),
                    'vel': n['vel']
                })

        for m_idx in range(1, num_measures + 1):
            measure = ET.SubElement(part, 'measure', number=str(m_idx))

            # Initial Measure Attributes
            if m_idx == 1:
                attributes = ET.SubElement(measure, 'attributes')
                ET.SubElement(attributes, 'divisions').text = str(ticks_per_beat)

                # Key signature
                key_elem = ET.SubElement(attributes, 'key')
                ET.SubElement(key_elem, 'fifths').text = str(fifths)
                ET.SubElement(key_elem, 'mode').text = mode

                # Time signature
                time_elem = ET.SubElement(attributes, 'time')
                ET.SubElement(time_elem, 'beats').text = str(time_sig_num)
                ET.SubElement(time_elem, 'beat-type').text = str(time_sig_denom)

                # Clef specification
                clef_elem = ET.SubElement(attributes, 'clef')
                clef_sign = ET.SubElement(clef_elem, 'sign')
                clef_line = ET.SubElement(clef_elem, 'line')

                if is_drums:
                    clef_sign.text = "percussion"
                    clef_line.text = "2"
                elif is_bass or is_piano_lh:
                    clef_sign.text = "F"
                    clef_line.text = "4"
                else:
                    clef_sign.text = "G"
                    clef_line.text = "2"

                # Metronome tempo mark
                direction = ET.SubElement(measure, 'direction', placement="above")
                direction_type = ET.SubElement(direction, 'direction-type')
                metronome = ET.SubElement(direction_type, 'metronome')
                ET.SubElement(metronome, 'beat-unit').text = "quarter"
                ET.SubElement(metronome, 'per-minute').text = f"{bpm:.1f}"
                sound = ET.SubElement(direction, 'sound')
                sound.set('tempo', str(int(round(bpm))))

            m_notes = measure_notes[m_idx]
            if not m_notes:
                # Whole measure rest
                note_elem = ET.SubElement(measure, 'note')
                ET.SubElement(note_elem, 'rest')
                ET.SubElement(note_elem, 'duration').text = str(ticks_per_bar)
                ET.SubElement(note_elem, 'voice').text = "1"
                ET.SubElement(note_elem, 'type').text = "whole" if time_sig_num == 4 else "half"
            else:
                # Sort measure notes by onset
                m_notes.sort(key=lambda x: (x['rel_start'], x['pitch']))
                prev_pos = 0

                for n_info in m_notes:
                    onset = n_info['rel_start']
                    dur = n_info['dur']

                    # Insert rest if there is a gap before note
                    if onset > prev_pos:
                        gap_dur = onset - prev_pos
                        rest_elem = ET.SubElement(measure, 'note')
                        ET.SubElement(rest_elem, 'rest')
                        ET.SubElement(rest_elem, 'duration').text = str(gap_dur)
                        ET.SubElement(rest_elem, 'voice').text = "1"
                        prev_pos = onset

                    # Write pitch note
                    note_elem = ET.SubElement(measure, 'note')
                    if onset < prev_pos:
                        # Simultaneous note (chord)
                        ET.SubElement(note_elem, 'chord')
                    else:
                        prev_pos = onset + dur

                    if is_drums:
                        unpitched = ET.SubElement(note_elem, 'unpitched')
                        ET.SubElement(unpitched, 'display-step').text = 'E'
                        ET.SubElement(unpitched, 'display-octave').text = '4'
                    else:
                        pitch_elem = ET.SubElement(note_elem, 'pitch')
                        p_val = n_info['pitch']
                        octave = (p_val // 12) - 1
                        step = _STEP_NAMES[p_val % 12]
                        alter = _ALTER_VALS[p_val % 12]

                        ET.SubElement(pitch_elem, 'step').text = step
                        if alter != 0:
                            ET.SubElement(pitch_elem, 'alter').text = str(alter)
                        ET.SubElement(pitch_elem, 'octave').text = str(octave)

                    ET.SubElement(note_elem, 'duration').text = str(max(1, dur))
                    ET.SubElement(note_elem, 'voice').text = "1"

                    # Calculate note type string
                    quarter_fraction = dur / ticks_per_beat
                    if quarter_fraction >= 3.5:
                        ntype = "whole"
                    elif quarter_fraction >= 1.75:
                        ntype = "half"
                    elif quarter_fraction >= 0.85:
                        ntype = "quarter"
                    elif quarter_fraction >= 0.4:
                        ntype = "eighth"
                    else:
                        ntype = "16th"
                    ET.SubElement(note_elem, 'type').text = ntype

                # Fill trailing rest if measure not full
                if prev_pos < ticks_per_bar:
                    tail_dur = ticks_per_bar - prev_pos
                    rest_elem = ET.SubElement(measure, 'note')
                    ET.SubElement(rest_elem, 'rest')
                    ET.SubElement(rest_elem, 'duration').text = str(tail_dur)
                    ET.SubElement(rest_elem, 'voice').text = "1"

    out_path = Path(output_xml_path)
    tree = ET.ElementTree(score)
    ET.indent(tree, space="  ", level=0)
    tree.write(str(out_path), encoding="utf-8", xml_declaration=True)
    return out_path


        print("Transcribing audio notes...")
        events = []
        start_t = time.time()
        real_stdout = sys.stdout

        # Strategy: High-Accuracy Demucs Stem Separation vs Direct Single-Pass
        if is_high_accuracy:
            try:
                two_stems_only = (instrument_focus == "Vocals Only (Clean Solo Voice)")
                stem_dir = separate_stems_demucs(wav_input, two_stems=two_stems_only)
                
                stem_jobs = []
                if two_stems_only:
                    stem_jobs.append((stem_dir / "vocals.wav", ["voice"]))
                else:
                    if (stem_dir / "vocals.wav").exists():
                        stem_jobs.append((stem_dir / "vocals.wav", ["voice"]))
                    if (stem_dir / "bass.wav").exists():
                        stem_jobs.append((stem_dir / "bass.wav", ["electric_bass"]))
                    if (stem_dir / "drums.wav").exists():
                        stem_jobs.append((stem_dir / "drums.wav", ["drums"]))
                    if (stem_dir / "other.wav").exists():
                        stem_jobs.append((stem_dir / "other.wav", ["acoustic_piano", "clean_electric_guitar", "distorted_electric_guitar"]))

                for s_wav, s_insts in stem_jobs:
                    if not s_wav.exists(): continue
                    print(f"  • Transcribing isolated stem: {s_wav.name}...")
                    with open(s_wav, "rb") as sf:
                        s_audio, s_sr = _read_wav_file(sf)
                    with torch.inference_mode():
                        for ev in model.transcribe((s_audio, s_sr), instruments=s_insts, batch_size=1):
                            if not isinstance(ev, ProgressEvent):
                                events.append(ev)
                    del s_audio
                    gc.collect()
            except Exception as demucs_err:
                print(f"⚠️ Demucs notice: {demucs_err} -> Falling back to direct mix transcription.")
                with open(wav_input, "rb") as f:
                    w_audio, w_sr = _read_wav_file(f)
                with torch.inference_mode():
                    for ev in model.transcribe((w_audio, w_sr), instruments=inst_list, batch_size=1):
                        if not isinstance(ev, ProgressEvent):
                            events.append(ev)
                del w_audio
        else:
            with open(wav_input, "rb") as f:
                w_audio, w_sr = _read_wav_file(f)
            with torch.inference_mode():
                for ev in model.transcribe((w_audio, w_sr), instruments=inst_list, batch_size=1):
                    if not isinstance(ev, ProgressEvent):
                        events.append(ev)
            del w_audio

        elapsed = time.time() - start_t
        print(f"\n\n✅ Extraction complete in {_fmt_time(elapsed)} — {len(events)} note events extracted.")

        # Resolve BPM (auto or manual)
        final_bpm = 120.0
        if "auto" in str(bpm).lower():
            try:
                print("Attempting AI beat tracking & tempo detection...")
                if hasattr(model, 'transcribe_and_postprocess'):
                    _, ai_grid = model.transcribe_and_postprocess(wav_input, detect_tempo="best-effort", quantize=False)
                    if ai_grid is not None and getattr(ai_grid, 'bpm', None):
                        final_bpm = float(ai_grid.bpm)
                        print(f"✓ AI Detected Tempo: {final_bpm:.1f} BPM")
            except Exception as tempo_err:
                print(f"⚠️ AI beat-tracking notice: {tempo_err} -> Using adaptive tempo {final_bpm:.1f} BPM")
        elif str(bpm).lower() == "custom":
            final_bpm = float(custom_bpm)
        else:
            try:
                final_bpm = float(bpm)
            except ValueError:
                final_bpm = 120.0

        raw_midi_bytes = model.events_to_midi_bytes(iter(events))
        raw_mid = mido.MidiFile(file=io.BytesIO(raw_midi_bytes))

        # Resolve Time Signature (auto or manual)
        req_num, req_denom = parse_time_signature_choice(time_signature)
        if req_num is not None:
            eff_ts_num, eff_ts_denom = req_num, req_denom
        else:
            eff_ts_num, eff_ts_denom = detect_time_sig_from_midi_notes(raw_mid)

        # Apply robust musical post-processing (Time Signature, Key Signature, Tempo, Single Voice & Quantization)
        print("Applying post-processing (Time Signature, Key Signature, Tempo, Single Voice & Quantization)...")
        processed_mid, final_midi_bytes, resolved_key = postprocess_midi_events(
            raw_mid,
            bpm_val=final_bpm,
            time_sig_num=eff_ts_num,
            time_sig_denom=eff_ts_denom,
            key_sig_choice=key_signature,
            quantize=enable_quantization,
            grid_division=grid_div,
            quant_strength=quantize_strength,
            fill_gaps=fill_gaps_beats,
            trim_lead_bars=trim_leading_bars,
            single_voice=True,
            separate_channels=separate_instrument_tracks
        )


